# 04 — Evaluation

**Goal:** Deep-dive into model performance.

Beyond accuracy:
- Confusion matrix (where errors happen)
- Precision / Recall / F1 (per class)
- Feature importance

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import joblib
from src.data_loader import load_raw_data
from src.preprocess import split_and_scale
from src.config import MODELS_DIR

df = load_raw_data(save_csv=False)
X_train, X_test, y_train, y_test, scaler, feature_names = split_and_scale(df)

model = joblib.load(MODELS_DIR / "svm_tuned.pkl")

## Confusion Matrix

Rows = actual, Columns = predicted.
Off-diagonal = misclassifications.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(cmap="Blues")
plt.title("Confusion Matrix — Tuned SVM")
plt.show()

**Observation:** 1 misclassification between versicolor & virginica — expected overlap.

## Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

| Metric | What it means |
|--------|---------------|
| **Precision** | Of predicted positives, how many correct? |
| **Recall** | Of actual positives, how many found? |
| **F1** | Harmonic mean of P & R |

## Feature Importance (Random Forest)

In [ ]:
import pandas as pd
import joblib

rf = joblib.load(MODELS_DIR / "random_forest.pkl")
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values()

imp.plot(kind="barh", color="teal", figsize=(7, 4))
plt.title("Feature Importance — Random Forest")
plt.show()

**Matches EDA:** Petal length & width dominate.

## Inference Demo

In [ ]:
from src.predict import predict

sample = [5.1, 3.5, 1.4, 0.2]
label, proba = predict(sample, model_name="svm_tuned")
print(f"Prediction: {label}")
print(f"Probabilities: {proba}")

## ✅ Final Summary

- **Best model:** Tuned SVM (~98% accuracy)
- **Key features:** Petal length & width
- **Weakness:** versicolor ↔ virginica confusion
- **Ready for:** deployment (`api/`, `app/`)

**Project complete!** 🎉